# Day 20 — Streaming Aggregations & Watermarks

**What you will learn:**
- Stateful aggregations: windowed counts, sums, and averages over event time
- Watermarks: how Spark decides when late data is too late
- Tumbling vs sliding windows
- `dropDuplicates` in streaming: deduplication with watermark
- `flatMapGroupsWithState` — arbitrary stateful logic

**Exam relevance:** Streaming (10%) — watermark semantics and window types are the core exam topics for stateful streaming.

In [ ]:
from pyspark.sql.functions import (
    col, window, count, sum, avg, round,
    current_timestamp, expr, to_timestamp, lit
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import tempfile, os, time

CHECKPOINT = tempfile.mkdtemp(prefix="stream_ckpt2_")
SOURCE_DIR = tempfile.mkdtemp(prefix="stream_src2_")

## 1. Event Time vs Processing Time

- **Event time**: when the event actually happened (embedded in the data)
- **Processing time**: when Spark processes it

For correct aggregations over time, always use **event time**.

Late data = event with an `event_time` older than what Spark has already processed.

## 2. Tumbling Windows

Non-overlapping, fixed-size time buckets.

```
│ 10:00–10:05 │ 10:05–10:10 │ 10:10–10:15 │
```

In [ ]:
# Seed: sales events with event_time
from datetime import datetime, timezone

seed_rows = [
    ("ORD-001", "EU", 150, "2024-06-01 10:01:00"),
    ("ORD-002", "US",  89, "2024-06-01 10:02:00"),
    ("ORD-003", "EU", 220, "2024-06-01 10:03:00"),
    ("ORD-004", "EU",  55, "2024-06-01 10:07:00"),
    ("ORD-005", "US", 310, "2024-06-01 10:08:00"),
    ("ORD-006", "EU",  99, "2024-06-01 10:12:00"),
    # Late event — event_time 10:03 but arrives later
    ("ORD-007", "EU",  44, "2024-06-01 10:03:30"),
]

schema = StructType([
    StructField("order_id",   StringType(),  True),
    StructField("region",     StringType(),  True),
    StructField("amount",     IntegerType(), True),
    StructField("event_time", StringType(),  True),
])

seed_df = spark.createDataFrame(seed_rows, schema=schema) \
    .withColumn("event_time", to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss"))

seed_df.write.mode("overwrite").json(SOURCE_DIR)
print("Seed data written.")

In [ ]:
stream_schema = StructType([
    StructField("order_id",   StringType(),  True),
    StructField("region",     StringType(),  True),
    StructField("amount",     IntegerType(), True),
    StructField("event_time", TimestampType(), True),
])

stream_df = spark.readStream \
    .schema(stream_schema) \
    .json(SOURCE_DIR)

# Tumbling window: 5-minute buckets — count orders and sum revenue per region
windowed = stream_df \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "5 minutes"),  # tumbling window
        col("region")
    ) \
    .agg(
        count("order_id").alias("order_count"),
        sum("amount").alias("total_revenue"),
    )

query = windowed.writeStream \
    .format("console") \
    .outputMode("update") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", os.path.join(CHECKPOINT, "tumbling")) \
    .option("truncate", False) \
    .start()

query.awaitTermination(timeout=60)
print("Tumbling window query done.")

## 3. Sliding Windows

Overlapping time windows — each event can belong to multiple windows.

```python
window(col("event_time"), "10 minutes", "5 minutes")
# windowDuration=10min, slideDuration=5min
# Windows: [10:00-10:10], [10:05-10:15], [10:10-10:20] ...
```

Use sliding windows for moving averages.

In [ ]:
# Sliding window: 10-minute window, slides every 5 minutes
sliding = stream_df \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "10 minutes", "5 minutes"),
        col("region")
    ) \
    .agg(round(avg("amount"), 2).alias("avg_amount"))

q2 = sliding.writeStream \
    .format("console") \
    .outputMode("update") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", os.path.join(CHECKPOINT, "sliding")) \
    .option("truncate", False) \
    .start()

q2.awaitTermination(timeout=60)
print("Sliding window query done.")

## 4. Watermarks — Handling Late Data

```python
.withWatermark("event_time", "10 minutes")
```

**What it means:**
- Spark tracks `max(event_time)` seen so far = watermark threshold
- Watermark = `max_event_time - delay` = 10 min in this case
- Events with `event_time < watermark` are **dropped** (too late)
- Windows with `end_time < watermark` are **finalised** and emitted

**Why watermarks are needed:**
- Without watermark, Spark must keep state forever (waiting for infinitely late data)
- Watermark bounds state size

> **Exam tip:** Watermark is required for `append` output mode with aggregation. Without watermark, `append` mode is not supported.

## 5. Streaming Deduplication

In [ ]:
# Drop duplicate order_ids within the watermark window
deduped = stream_df \
    .withWatermark("event_time", "10 minutes") \
    .dropDuplicates(["order_id", "event_time"])

q3 = deduped.writeStream \
    .format("console") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", os.path.join(CHECKPOINT, "dedup")) \
    .start()

q3.awaitTermination(timeout=60)
print("Dedup query done.")

## 6. Output Mode × Aggregation Compatibility

| Scenario | `append` | `complete` | `update` |
|---|---|---|---|
| No aggregation | ✅ | ❌ | ✅ |
| Aggregation, no watermark | ❌ | ✅ | ✅ |
| Aggregation + watermark | ✅ | ✅ | ✅ |
| `dropDuplicates` | ✅ (with watermark) | ❌ | ❌ |

> **Exam tip:** `complete` mode rewrites ALL results every batch — memory grows with result set size. Use `update` or `append` for production.

---

## Day 20 Checklist

- [ ] Used `window()` for a 5-minute tumbling window on event time
- [ ] Used `window()` with two durations for a sliding window
- [ ] Applied `withWatermark()` — understand that it bounds state and allows late-data drops
- [ ] Know the watermark formula: `watermark = max(event_time) - delay`
- [ ] Used `dropDuplicates` with watermark in streaming
- [ ] Know the output mode × aggregation compatibility matrix

**Next:** Day 21 — Streaming Lab (end-to-end streaming pipeline)